# Context Engineering

**WatSPEED Agentic AI prep — Week 2-3 - context engineering**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## Context engineering

A named learning outcome of the course, and the skill that separates agents that work
from agents that almost work. The model has **no memory** — it sees exactly the list
of messages you send, every single call. Context engineering is deciding what goes in
that list and what gets left out.

Three pressures pull against each other:

| Pressure | Symptom when you get it wrong |
|---|---|
| Fit the window | Truncation errors, or silent dropping of the earliest turns |
| Keep what matters | Agent "forgets" a constraint you gave it in step 1 |
| Keep it cheap | Token bill grows quadratically over a long run |

In [2]:
def approx_tokens(messages) -> int:
    """Rough count: ~4 characters per token. Good enough for budgeting."""
    return sum(len(str(m.get("content", ""))) for m in messages) // 4

conversation = [{"role": "system", "content": "You are a survey analysis assistant. Always cite N."}]
for i in range(1, 13):
    conversation.append({"role": "user", "content": f"Question {i}: break down trust by variable {i}."})
    conversation.append({"role": "assistant", "content": f"Result {i}: mean trust 3.1, N=2041, p<0.05. " * 6})

print(f"{len(conversation)} messages, ~{approx_tokens(conversation)} tokens")

25 messages, ~903 tokens


### Strategy 1 — sliding window

Keep the system prompt plus the last *k* turns. Cheap and simple. The failure mode is
obvious once you see it: instructions from early turns fall off the edge.

In [3]:
def sliding_window(messages, keep: int = 6):
    system = [m for m in messages if m["role"] == "system"]
    return system + messages[-keep:]

windowed = sliding_window(conversation)
print(f"{len(windowed)} messages, ~{approx_tokens(windowed)} tokens "
      f"({100 - approx_tokens(windowed) * 100 // approx_tokens(conversation)}% smaller)")
show("First kept turn", windowed[1]["content"][:80])
print("\nNote: questions 1-9 are gone. If Q2 set a constraint, it is now invisible.")

7 messages, ~240 tokens (74% smaller)
First kept turn:
  Question 10: break down trust by variable 10.

Note: questions 1-9 are gone. If Q2 set a constraint, it is now invisible.


### Strategy 2 — compaction

Summarise the old turns into one message instead of dropping them. This is what
Claude Code and most production agents do when a session gets long.

In [4]:
def compact(messages, keep_recent: int = 4):
    system = [m for m in messages if m["role"] == "system"]
    body = [m for m in messages if m["role"] != "system"]
    old, recent = body[:-keep_recent], body[-keep_recent:]
    if not old:
        return messages
    facts = [m["content"].split(":")[0] for m in old if m["role"] == "user"]
    summary = {"role": "system",
               "content": "Earlier in this session the user asked about: " + "; ".join(facts) + "."}
    return system + [summary] + recent

compacted = compact(conversation)
print(f"{len(compacted)} messages, ~{approx_tokens(compacted)} tokens")
show("Compaction message", compacted[1]["content"])

6 messages, ~205 tokens
Compaction message:
  Earlier in this session the user asked about: Question 1; Question 2; Question 3; Question 4; Question 5; Question 6; Question 7; Question 8; Question 9; Question 10.


### Strategy 3 — structured output

Free-text answers are expensive to carry and impossible to assert on. Forcing the model
to return a schema shrinks context *and* makes the result testable — which is how you
put an agent in a pipeline instead of a chat window.

In [5]:
from pydantic import BaseModel, Field

class Finding(BaseModel):
    variable: str
    effect: str = Field(description="direction of association")
    p_value: float
    n: int

    def one_line(self) -> str:
        sig = "significant" if self.p_value < 0.05 else "not significant"
        return f"{self.variable}: {self.effect} ({sig}, p={self.p_value}, N={self.n})"

raw = '{"variable": "age_group", "effect": "trust rises with age", "p_value": 0.006, "n": 2041}'
finding = Finding.model_validate_json(raw)

show("Parsed and validated", finding.model_dump())
print("\nCompact form for context:", finding.one_line())
print(f"{len(raw)} chars of JSON -> {len(finding.one_line())} chars carried forward")

Parsed and validated:
  {
    "variable": "age_group",
    "effect": "trust rises with age",
    "p_value": 0.006,
    "n": 2041
  }

Compact form for context: age_group: trust rises with age (significant, p=0.006, N=2041)
88 chars of JSON -> 62 chars carried forward


### Putting a budget on it

In production you don't choose one strategy — you apply them by budget.

In [6]:
def fit_to_budget(messages, budget_tokens: int = 400):
    for strategy in (lambda m: m, compact, sliding_window):
        candidate = strategy(messages)
        if approx_tokens(candidate) <= budget_tokens:
            print(f"{strategy.__name__ if hasattr(strategy,'__name__') else 'as-is':<16} "
                  f"fits: ~{approx_tokens(candidate)} tokens")
            return candidate
    print("Nothing fits - drop tool results or summarise harder.")
    return sliding_window(messages, keep=2)

banner("Budget 400 tokens")
_ = fit_to_budget(conversation, 400)
banner("Budget 5000 tokens")
_ = fit_to_budget(conversation, 5000)


Budget 400 tokens
compact          fits: ~205 tokens

Budget 5000 tokens
<lambda>         fits: ~903 tokens


---
### Try it yourself

1. `compact()` keeps only user questions. Rewrite it to preserve any message containing
   a number, then compare token counts.
2. Add a `pinned` flag so a message can never be evicted. Which of your messages need it?
3. Measure: at what conversation length does compaction beat windowing on fidelity?